# Master Benchmarking Suite: 20 Advanced Evolutionary Architectures (Model 1 -> Model 20)
## Targeted for Maximum Accuracy & High-Speed Deployment on NVIDIA Jetson Edge AI Devices

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/doantrongthai/flood_mamba_edge/blob/main/screening_20_jetson_models.ipynb)

This notebook evaluates 20 systematically engineered architectures built from the Round 1 Champion (**Model 12: 0.8076 mIoU on FloodKaggle**), pushing for state-of-the-art accuracy across both Satellite (`floodkaggle`) and UAV (`aifloodsense`) domains while remaining strictly lightweight (<0.9M parameters) and Jetson deployable (FP32/FP16 native, Tensor Core aligned channels, zero-copy PixelShuffle, and structural reparameterization).

### The 20 Models:
- **Model 1 (`model1`)**: **Champion Baseline (Round 1 Winner / Model 12 Architecture)** - OmniASDBR + Scale-Aware Mamba + SpatialBoundaryPS + Laplacian Guidance + Aux Head
- **Model 2 (`model2`)**: **InvRes-Omni Super-Aqua** - Fusing 2x Inverted Residual channel expansion from Model 4 with Omni-ASDConv to boost fine UAV flood textures
- **Model 3 (`model3`)**: **Direct Laplacian-Coupled Decoder** - Direct physical Laplacian ∇²X boundary prior coupled into all decoder spatial gates
- **Model 4 (`model4`)**: **Dual-Stage Deep Supervision** - Dual auxiliary prediction heads at d3 (32x32) and d2 (64x64)
- **Model 5 (`model5`)**: **RepOmni Structural Reparameterization** - Multi-branch strip conv in training -> Fused single 7x7 depthwise on Jetson
- **Model 6 (`model6`)**: **Topographic Coordinate-Aware SGR** - Injects normalized (x, y) spatial coordinates for elevation/gravity-aware water flow modeling
- **Model 7 (`model7`)**: **4-Way Bidirectional Cross-Scan Mamba** - Horizontal and Vertical bidirectional 2D omni-scanning
- **Model 8 (`model8`)**: **Haar Wavelet High-Frequency Skip** - 2D Haar DWT high-pass subbands (LH, HL, HH) bypass downsampling into decoder
- **Model 9 (`model9`)**: **Asymmetric Strip-PPM Bottleneck** - Horizontal (1xW) and Vertical (Hx1) strip pooling pyramid for wide water bodies and corridors
- **Model 10 (`model10`)**: **Dynamic Feature Alignment Decoder (DFAD)** - Learns spatial deformation offsets to align skip features with upsampled features
- **Model 11 (`model11`)**: **Content-Aware Boundary Modulated Attention (CABMA)** - Local intensity variance gating to filter out reflections and glares
- **Model 12 (`model12`)**: **Multi-Dilated Strip Pyramid Block (DilPyramid)** - Parallel strip convolutions with dilations (d=1, 2, 3)
- **Model 13 (`model13`)**: **Differentiable Morphological Boundary Refiner** - Soft morphological erosion/dilation transition zone refiner
- **Model 14 (`model14`)**: **Edge-Enhanced Selective Feature Fusion (E-SFF)** - Channel cross-attention skip modulation
- **Model 15 (`model15`)**: **ConvNeXt-Style Inverted Omni-ASD Block (OmniNeXt)** - 7x7 depthwise + LayerNorm + 2x inverted expansion + Omni-ASD
- **Model 16 (`model16`)**: **InvRes-Omni + Direct Laplacian Decoder** - High UAV texture capacity + physical ∇²X prior
- **Model 17 (`model17`)**: **InvRes-Omni + Dual-Stage Deep Supervision** - High UAV texture capacity + dual-resolution gradient flow
- **Model 18 (`model18`)**: **Direct Laplacian + Dual-Stage Deep Supervision** - Direct ∇²X prior + dual-resolution gradient flow
- **Model 19 (`model19`)**: **All-In-One Unified Champion** - InvRes-Omni + Direct Laplacian + MultiDil-SGR + Dual Aux + MorphoRefiner
- **Model 20 (`model20`)**: **Jetson Streamlined TensorRT Champion** - RepOmni reparameterization + Tensor Core aligned + Dual Aux + switch_to_deploy()

### Experimental Protocols:
- **Image Resolution**: $128 \times 128$
- **Batch Size**: 4
- **Epochs**: 100
- **Loss**: `lovasz_focal`
- **Learning Rate**: 0.001
- **Seed**: 42
- **Datasets**: `floodkaggle`, `aifloodsense`

In [ ]:
# 1. Environment & GPU Hardware Verification
!nvidia-smi
import torch
print(f"PyTorch Version : {torch.__version__}")
print(f"CUDA Available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name     : {torch.cuda.get_device_name(0)}")
    print(f"Capability      : {torch.cuda.get_device_capability(0)}")

In [ ]:
# 2. Setup Official Hardware-Accelerated Mamba CUDA Kernel on Google Colab GPU (Tesla T4)
import zipfile, json, sys, sysconfig, pathlib, subprocess, torch

SP = pathlib.Path(sysconfig.get_paths()["purelib"])
FILE_ID = "1kuBAKlNtIc8G1KQFqO8xV7t9ZEM4XATB"

assert torch.cuda.get_device_capability(0) == (7, 5), f"Vui long chon Colab GPU la Tesla T4 (capability 7.5), hien tai: {torch.cuda.get_device_name(0)}"

# 1) Tai goi pre-compiled CUDA wheel
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gdown"])
import gdown
gdown.download(id=FILE_ID, output="/content/pack.zip", quiet=False)

# 2) Giai nen vao site-packages
with zipfile.ZipFile("/content/pack.zip") as z:
    assert z.testzip() is None, "File tai ve bi hong, vui long tai lai"
    m = json.loads(z.read("__manifest__.json"))
    now = {"python": sys.version.split()[0], "torch": torch.__version__, "cuda": torch.version.cuda}
    bad = {k: (m[k], now[k]) for k in now if m[k] != now[k]}
    assert not bad, f"Moi truong Colab khong khop: {bad}"
    for name in z.namelist():
        if not name.startswith("__"):
            z.extract(name, SP)
print("Da giai nen vao:", SP)

# 3) Cai cac dependency bo tro
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "einops", "packaging", "ninja", "transformers", "triton"])

# 4) Kiem tra CUDA kernel Mamba tren GPU
import selective_scan_cuda, causal_conv1d_cuda
from mamba_ssm.modules.mamba_simple import causal_conv1d_fn
from mamba_ssm import Mamba
print("causal_conv1d_fn:", "OK (CUDA kernel)" if causal_conv1d_fn is not None else "FALLBACK")
y = Mamba(d_model=64).cuda()(torch.randn(2, 128, 64, device="cuda"))
y.mean().backward()
print("[OK] OFFICIAL MAMBA HARDWARE CUDA KERNEL ACTIVE:", y.shape)

In [ ]:
# 3. Clone / Update Repository & Install Utilities
import os, sys

if not os.path.exists("flood_mamba_edge"):
    !git clone https://github.com/doantrongthai/flood_mamba_edge.git
    %cd flood_mamba_edge
else:
    %cd flood_mamba_edge
    !git pull origin main

!pip install -q tabulate pandas gdown
print("[OK] Workspace ready at:", os.getcwd())

In [ ]:
# 4. Verify & Pre-Download Datasets
import sys, subprocess

DATASETS = ["floodkaggle", "aifloodsense"]
for ds in DATASETS:
    print(f"\n--- Checking / Downloading Dataset: {ds} ---")
    subprocess.run([
        sys.executable, "main.py",
        "--dataset", ds,
        "--download",
        "--epochs", "1",
        "--batch_size", "2",
        "--model", "model1",
        "--size", "128"
    ], check=False)
print("\n[OK] All datasets downloaded and verified successfully!")

In [ ]:
# 5. Master Automated Benchmark Loop with Progress Bar Noise Filter
# Suppresses verbose tqdm batch output to avoid Colab 5,000-line buffer truncation
import sys, os, re, time, subprocess
import pandas as pd
from tabulate import tabulate

SEED = 42
IMAGE_SIZE = 128
BATCH_SIZE = 4
EPOCHS = 100
LR = 0.001
LOSS_NAME = "lovasz_focal"
NUM_CLASSES = 1
DATASETS = ["floodkaggle", "aifloodsense"]

MODELS_TO_BENCHMARK = [f"model{i}" for i in range(1, 21)]

total_runs = len(MODELS_TO_BENCHMARK) * len(DATASETS)
run_idx = 0
results = []
csv_out_path = "screening_20_models_results.csv"

# Load previous partial results if resuming
if os.path.exists(csv_out_path):
    try:
        prev_df = pd.read_csv(csv_out_path)
        results = prev_df.to_dict("records")
        print(f"[RESUME] Loaded {len(results)} completed runs from {csv_out_path}")
    except Exception as e:
        print(f"[RESUME WARNING] Could not read existing results: {e}")

completed_keys = {(r["model"], r["dataset"]) for r in results if "model" in r and "dataset" in r}

print(f"Total planned runs: {total_runs} (Already completed: {len(completed_keys)})")
print("=" * 85)

for model_name in MODELS_TO_BENCHMARK:
    for ds in DATASETS:
        run_idx += 1
        if (model_name, ds) in completed_keys:
            print(f"[{run_idx}/{total_runs}] [SKIP] {model_name} on {ds} already completed.")
            continue

        print(f"\n{'='*85}")
        print(f"[{run_idx}/{total_runs}] STARTING: {model_name.upper()} on {ds.upper()} (Seed={SEED}, Epochs={EPOCHS})")
        print(f"{'='*85}")

        cmd = [
            sys.executable, "-u", "main.py",
            "--dataset", ds,
            "--model", model_name,
            "--loss", LOSS_NAME,
            "--epochs", str(EPOCHS),
            "--batch_size", str(BATCH_SIZE),
            "--size", str(IMAGE_SIZE),
            "--lr", str(LR),
            "--seed", str(SEED),
            "--output_path", "outputs"
        ]

        t0 = time.time()
        proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)

        captured_output = []
        for line in proc.stdout:
            # FILTER OUT batch-level tqdm lines to avoid Colab 5,000-line buffer truncation
            if "%|" in line or "[Train]:" in line or "[Val]:" in line or "it/s" in line or "\r" in line:
                continue
            sys.stdout.write(line)
            sys.stdout.flush()
            captured_output.append(line)

        proc.wait()
        elapsed = time.time() - t0
        full_text = "".join(captured_output)

        # Parse metrics
        m_miou = re.search(r"mIoU:\s+([0-9\.]+)", full_text)
        m_dice = re.search(r"Dice / F1 Score:\s+([0-9\.]+)", full_text)
        m_prec = re.search(r"Precision:\s+([0-9\.]+)", full_text)
        m_rec = re.search(r"Recall:\s+([0-9\.]+)", full_text)
        m_biou = re.search(r"Boundary IoU:\s+([0-9\.]+)", full_text)
        m_loss = re.search(r"Best Val Loss:\s+([0-9\.]+)", full_text)
        m_params = re.search(r"Total Params \(Train\):\s+([0-9,]+)", full_text)
        m_fps = re.search(r"FPS:\s+([0-9\.]+)", full_text)

        row = {
            "model": model_name,
            "dataset": ds,
            "mIoU": float(m_miou.group(1)) if m_miou else None,
            "Dice": float(m_dice.group(1)) if m_dice else None,
            "Precision": float(m_prec.group(1)) if m_prec else None,
            "Recall": float(m_rec.group(1)) if m_rec else None,
            "B_IoU": float(m_biou.group(1)) if m_biou else None,
            "val_loss": float(m_loss.group(1)) if m_loss else None,
            "params": m_params.group(1) if m_params else None,
            "fps": float(m_fps.group(1)) if m_fps else None,
            "time_sec": round(elapsed, 2)
        }
        results.append(row)

        # Save to CSV after every run
        df_current = pd.DataFrame(results)
        df_current.to_csv(csv_out_path, index=False)

        print(f"\n[DONE] {model_name} on {ds} in {elapsed:.1f}s | mIoU: {row['mIoU']} | Dice: {row['Dice']} | B-IoU: {row['B_IoU']}")

print("\n\n" + "=" * 90)
print("=== FINAL COMPLETE 20-MODEL MASTER BENCHMARK SUMMARY ===")
print("=" * 90)

df_all = pd.DataFrame(results)
print(tabulate(df_all, headers="keys", tablefmt="github", floatfmt=".4f"))

# Dataset comparison pivot table
pivot = df_all.pivot_table(index="model", columns="dataset", values="mIoU")
print("\n=== mIoU COMPARISON ACROSS DATASETS ===")
print(tabulate(pivot, headers="keys", tablefmt="github", floatfmt=".4f"))

In [ ]:
# 6. Export Results to CSV & Download
import pandas as pd
from google.colab import files

df_all = pd.DataFrame(results)
df_all.to_csv("master_evolution_20_models_results.csv", index=False)
print("Saved to master_evolution_20_models_results.csv")
files.download("master_evolution_20_models_results.csv")